# Avance 2 — Ingeniería de características y análisis estadístico NIST

## Frente 3: Datos experimentales de Marshall y NIST

### Objetivo

Desarrollar y justificar técnicas de ingeniería de características para los datos experimentales del NIST, considerando la estructura de las mediciones, las variables físicas disponibles y las limitaciones del tamaño muestral.

El análisis se centra inicialmente en los 24 registros digitalizados de la Figura 6 de Marshall, correspondientes a ocho participantes y tres longitudes de voladizo (200, 300 y 400 µm). Se evalúan transformaciones de variables, codificación categórica, redundancia, posibles fugas de información y supuestos estadísticos.

Los resultados se interpretan como evidencia exploratoria para apoyar las decisiones de preparación de datos dentro de CRISP-ML(Q), sin asumir que las asociaciones observadas representan relaciones causales ni que las transformaciones propuestas constituyen un modelo predictivo validado.

In [23]:

# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from scipy.stats import f, shapiro, ttest_ind
from statsmodels.stats.anova import AnovaRM


## 2. Carga y validación de los datos

Se utiliza el conjunto de datos digitalizado de la Figura 6 de Marshall, disponible en el repositorio del proyecto. Este contiene mediciones del módulo de Young (`E_GPa`) para diferentes longitudes de voladizo (`L_um`), participantes, chips e instrumentos.

Antes de aplicar técnicas de ingeniería de características, se verifica la integridad estructural del conjunto de datos, incluyendo dimensiones, columnas requeridas, valores faltantes y duplicados.

La validación inicial permite identificar posibles problemas de calidad sin modificar los registros originales.

In [4]:

# ============================================================
# 2. CARGA Y VALIDACIÓN DE DATOS
# ============================================================

data_url = (
    "https://raw.githubusercontent.com/"
    "isabellaarangor/proyecto-integrador-team39-MNA/"
    "main/datos/nist/digitalizados/"
    "marshall_F6_reproducibilidad.csv"
)

df_marshall = pd.read_csv(data_url, comment="#")

required_columns = {
    "participant",
    "chip",
    "instrument",
    "L_um",
    "E_GPa",
    "x_px",
    "y_px",
}

missing_columns = required_columns - set(df_marshall.columns)

if missing_columns:
    raise ValueError(
        f"Faltan columnas requeridas: {sorted(missing_columns)}"
    )

print(f"Dimensiones: {df_marshall.shape}")
print(f"Registros duplicados: {df_marshall.duplicated().sum()}")

print("\nTipos de datos:")
display(df_marshall.dtypes.to_frame(name="Tipo"))

print("\nValores faltantes:")
display(df_marshall.isna().sum().to_frame(name="Faltantes"))

print("\nPrimeros registros:")
display(df_marshall.head(10))


Dimensiones: (24, 7)
Registros duplicados: 0

Tipos de datos:


,Tipo
participant,int64
chip,object
instrument,object
L_um,int64
E_GPa,float64
x_px,float64
y_px,float64



Valores faltantes:


,Faltantes
participant,0
chip,0
instrument,6
L_um,0
E_GPa,0
x_px,0
y_px,0



Primeros registros:


,participant,chip,instrument,L_um,E_GPa,x_px,y_px
0,1,chip 1,vibrómetro de doble haz,200,60.32,552.5,268.0
1,1,chip 1,vibrómetro de doble haz,300,65.61,560.9,244.1
2,1,chip 1,vibrómetro de doble haz,400,68.06,567.4,233.1
3,2,chip 1,vibrómetro de un haz,200,60.32,589.0,268.0
4,2,chip 1,vibrómetro de un haz,300,65.62,597.1,244.1
5,2,chip 1,vibrómetro de un haz,400,66.84,604.1,238.6
6,3,chip 1,interferómetro estroboscópico,200,60.31,626.0,268.1
7,3,chip 1,interferómetro estroboscópico,300,65.31,633.8,245.5
8,3,chip 1,interferómetro estroboscópico,400,67.37,640.8,236.2
9,4,chip 2,excitación PZT,200,58.30,670.1,277.1


## 3. Preparación e ingeniería de características

### 3.1. Tratamiento de valores faltantes y codificación categórica

Se genera una copia del conjunto de datos original para preservar su integridad. Los valores faltantes de la variable `instrument` se identifican mediante la categoría «No especificado», evitando atribuir un instrumento sin evidencia.

Las variables nominales `chip` e `instrument` se codifican mediante *one-hot encoding*, ya que sus categorías no poseen un orden natural.

Se conservan inicialmente los identificadores y las coordenadas de digitalización para fines de trazabilidad. Sin embargo, estas variables no se incorporarán a la matriz predictora final, pues no representan propiedades físicas independientes del experimento.

In [5]:

# ============================================================
# 3.1. LIMPIEZA Y CODIFICACIÓN CATEGÓRICA
# ============================================================

df_fe = df_marshall.copy()

# Identificar instrumentos sin información disponible.
df_fe["instrument"] = df_fe["instrument"].fillna(
    "No especificado"
)

# Codificar variables categóricas nominales.
df_encoded = pd.get_dummies(
    df_fe,
    columns=["chip", "instrument"],
    dtype=int,
)

print(f"Dimensiones originales: {df_marshall.shape}")
print(f"Dimensiones codificadas: {df_encoded.shape}")

print("\nValores faltantes después del tratamiento:")
display(df_encoded.isna().sum().to_frame(name="Faltantes"))

print("\nPrimeros registros codificados:")
display(df_encoded.head())


Dimensiones originales: (24, 7)
Dimensiones codificadas: (24, 15)

Valores faltantes después del tratamiento:


,Faltantes
participant,0
L_um,0
E_GPa,0
x_px,0
y_px,0
chip_chip 1,0
chip_chip 2,0
chip_chip 3,0
chip_chip 4,0
instrument_No especificado,0



Primeros registros codificados:


,participant,L_um,E_GPa,x_px,y_px,chip_chip 1,chip_chip 2,chip_chip 3,chip_chip 4,instrument_No especificado,instrument_excitación PZT,instrument_excitación térmica,instrument_interferómetro estroboscópico,instrument_vibrómetro de doble haz,instrument_vibrómetro de un haz
0,1,200,60.32,552.5,268.0,1,0,0,0,0,0,0,0,1,0
1,1,300,65.61,560.9,244.1,1,0,0,0,0,0,0,0,1,0
2,1,400,68.06,567.4,233.1,1,0,0,0,0,0,0,0,1,0
3,2,200,60.32,589.0,268.0,1,0,0,0,0,0,0,0,0,1
4,2,300,65.62,597.1,244.1,1,0,0,0,0,0,0,0,0,1


### 3.2. Generación de características derivadas

A partir de la longitud del voladizo (`L_um`) se generan tres representaciones adicionales:

- `L_mm`: longitud expresada en milímetros, como conversión de unidades.
- `L_squared`: longitud elevada al cuadrado, para explorar posibles relaciones no lineales.
- `L_inverse`: inverso de la longitud, para explorar relaciones inversas.

Estas transformaciones permiten evaluar distintas representaciones matemáticas de la longitud, pero no constituyen mediciones físicas independientes. Por ello, se analizará su redundancia antes de seleccionar las características que podrían utilizarse en etapas posteriores de modelado.

La inclusión de estas variables es exploratoria y no implica que todas deban conservarse simultáneamente.

In [6]:

# ============================================================
# 3.2. GENERACIÓN DE CARACTERÍSTICAS DERIVADAS
# ============================================================

df_encoded["L_mm"] = df_encoded["L_um"] / 1000.0
df_encoded["L_squared"] = df_encoded["L_um"] ** 2
df_encoded["L_inverse"] = 1.0 / df_encoded["L_um"]

derived_columns = [
    "L_um",
    "L_mm",
    "L_squared",
    "L_inverse",
    "E_GPa",
]

print("Características derivadas de la longitud:")
display(df_encoded[derived_columns].head(10))

print(f"\nDimensiones del conjunto: {df_encoded.shape}")


Características derivadas de la longitud:


,L_um,L_mm,L_squared,L_inverse,E_GPa
0,200,0.2,40000,0.005000,60.32
1,300,0.3,90000,0.003333,65.61
2,400,0.4,160000,0.002500,68.06
3,200,0.2,40000,0.005000,60.32
4,300,0.3,90000,0.003333,65.62
5,400,0.4,160000,0.002500,66.84
6,200,0.2,40000,0.005000,60.31
7,300,0.3,90000,0.003333,65.31
8,400,0.4,160000,0.002500,67.37
9,200,0.2,40000,0.005000,58.30



Dimensiones del conjunto: (24, 18)


### 3.3. Selección de características y prevención de fuga de información

Antes de realizar el escalamiento, se excluyen las variables que no deben formar parte de la matriz predictora:

- `participant`: identificador utilizado para reconocer las mediciones repetidas de cada participante.
- `x_px` y `y_px`: coordenadas provenientes del proceso de digitalización de la figura. No representan mediciones físicas independientes y podrían introducir información artificial sobre la variable objetivo.
- `L_mm`: conversión lineal de `L_um`, por lo que aporta información redundante.

La variable `E_GPa` se separa como objetivo del análisis.

Se conservan provisionalmente `L_um`, `L_squared` y `L_inverse`, junto con las categorías codificadas, para examinar su utilidad. Las transformaciones de longitud están matemáticamente relacionadas y no deben interpretarse como características independientes.

Esta selección es exploratoria. En un modelo predictivo posterior será necesario definir una estrategia de validación que respete la estructura de mediciones repetidas y ajustar cualquier transformación aprendida únicamente con los datos de entrenamiento.

In [7]:

# ============================================================
# 3.3. SELECCIÓN DE CARACTERÍSTICAS
# ============================================================

excluded_columns = [
    "participant",
    "x_px",
    "y_px",
    "L_mm",
    "E_GPa",
]

# Separar la variable objetivo sin modificar los datos originales.
y_selected = df_encoded["E_GPa"].copy()

# Excluir identificadores, coordenadas y variables redundantes.
X_selected = df_encoded.drop(
    columns=excluded_columns
).copy()

# Verificar la selección.
assert "E_GPa" not in X_selected.columns
assert "y_px" not in X_selected.columns
assert X_selected.isna().sum().sum() == 0

print(f"Dimensiones de X: {X_selected.shape}")
print(f"Dimensiones de y: {y_selected.shape}")

print("\nCaracterísticas seleccionadas:")
for column in X_selected.columns:
    print(f"- {column}")

print("\nPrimeros registros:")
display(X_selected.head())


Dimensiones de X: (24, 13)
Dimensiones de y: (24,)

Características seleccionadas:
- L_um
- chip_chip 1
- chip_chip 2
- chip_chip 3
- chip_chip 4
- instrument_No especificado
- instrument_excitación PZT
- instrument_excitación térmica
- instrument_interferómetro estroboscópico
- instrument_vibrómetro de doble haz
- instrument_vibrómetro de un haz
- L_squared
- L_inverse

Primeros registros:


,L_um,chip_chip 1,chip_chip 2,chip_chip 3,chip_chip 4,instrument_No especificado,instrument_excitación PZT,instrument_excitación térmica,instrument_interferómetro estroboscópico,instrument_vibrómetro de doble haz,instrument_vibrómetro de un haz,L_squared,L_inverse
0,200,1,0,0,0,0,0,0,0,1,0,40000,0.005000
1,300,1,0,0,0,0,0,0,0,1,0,90000,0.003333
2,400,1,0,0,0,0,0,0,0,1,0,160000,0.002500
3,200,1,0,0,0,0,0,0,0,0,1,40000,0.005000
4,300,1,0,0,0,0,0,0,0,0,1,90000,0.003333


### 3.4. Análisis de correlaciones y redundancia

Se calcula el coeficiente de correlación de Pearson entre las características numéricas seleccionadas y el módulo de Young (`E_GPa`) para identificar asociaciones lineales exploratorias.

Adicionalmente, se identifican pares de características con correlación absoluta superior a 0.95, como indicador de posible redundancia.

Es importante considerar que las transformaciones `L_um`, `L_squared` y `L_inverse` provienen de la misma variable original. Por tanto, una correlación elevada entre ellas es esperable y no implica que aporten información independiente.

Dado que las 24 observaciones incluyen mediciones repetidas de ocho participantes, las correlaciones se interpretan descriptivamente y no como evidencia inferencial independiente ni como criterios definitivos de selección para un modelo predictivo.

In [8]:

# ============================================================
# 3.4. CORRELACIONES Y REDUNDANCIA
# ============================================================

# Correlaciones de las características con la variable objetivo.
correlations = X_selected.corrwith(y_selected).dropna()

correlations = correlations.reindex(
    correlations.abs().sort_values(ascending=False).index
)

correlation_summary = pd.DataFrame(
    {
        "Característica": correlations.index,
        "Correlación Pearson": correlations.values,
        "Correlación absoluta": correlations.abs().values,
    }
)

print("Correlaciones con el módulo de Young:")
display(correlation_summary.round(4))

# Identificar pares de características altamente correlacionadas.
correlation_matrix = X_selected.corr().abs()
redundant_pairs = []

for i, column_a in enumerate(correlation_matrix.columns):
    for column_b in correlation_matrix.columns[i + 1:]:
        correlation = correlation_matrix.loc[column_a, column_b]

        if correlation > 0.95:
            redundant_pairs.append(
                {
                    "Variable 1": column_a,
                    "Variable 2": column_b,
                    "Correlación": round(correlation, 4),
                }
            )

print("\nPares con correlación absoluta superior a 0.95:")

if redundant_pairs:
    display(pd.DataFrame(redundant_pairs))
else:
    print("No se identificaron pares que superen el umbral.")


Correlaciones con el módulo de Young:


,Característica,Correlación Pearson,Correlación absoluta
0,L_inverse,-0.9020,0.9020
1,L_um,0.8822,0.8822
2,L_squared,0.8601,0.8601
3,chip_chip 1,0.3689,0.3689
4,chip_chip 2,-0.3183,0.3183
5,instrument_excitación PZT,-0.2451,0.2451
6,instrument_vibrómetro de doble haz,0.2077,0.2077
7,instrument_interferómetro estroboscópico,0.1701,0.1701
8,instrument_vibrómetro de un haz,0.1622,0.1622
9,instrument_excitación térmica,-0.1451,0.1451



Pares con correlación absoluta superior a 0.95:


,Variable 1,Variable 2,Correlación
0,L_um,L_squared,0.9954
1,L_um,L_inverse,0.9820
2,L_squared,L_inverse,0.9594


#### Interpretación y decisiones sobre las características

El análisis de correlación de Pearson muestra que las variables derivadas de la longitud presentan las asociaciones de mayor magnitud con el módulo de Young (`E_GPa`). La correlación más intensa corresponde a `L_inverse` (r = −0.9020), seguida de `L_um` (r = 0.8822) y `L_squared` (r = 0.8601).

Sin embargo, las tres representaciones de longitud presentan una redundancia elevada: las correlaciones absolutas entre ellas oscilan entre 0.9594 y 0.9954. Este resultado es consistente con que todas provienen de una misma variable física.

Las variables categóricas codificadas de chip e instrumento presentan asociaciones individuales de menor magnitud. Estas correlaciones no permiten establecer efectos independientes, particularmente porque las categorías pueden estar relacionadas con los participantes y las condiciones experimentales.

**Decisión de ingeniería de características:** se propone conservar `L_um` como representación principal de longitud en una matriz predictora inicial, manteniendo `L_inverse` y `L_squared` únicamente como alternativas para experimentos posteriores. No se recomienda incorporar las tres simultáneamente sin una justificación específica.

**Limitaciones:** los resultados son descriptivos y se calcularon sobre 24 mediciones correspondientes a ocho participantes. La dependencia entre observaciones impide interpretar estas correlaciones como pruebas inferenciales basadas en 24 muestras independientes. Además, la asociación estadística no demuestra causalidad ni garantiza capacidad de generalización.

## 4. Análisis estadístico de los datos experimentales

### 4.1. Estructura de las mediciones repetidas

Antes de realizar pruebas inferenciales, se examina la organización experimental de los datos digitalizados de Marshall.

El conjunto contiene 24 observaciones correspondientes a ocho participantes, con tres mediciones por participante para longitudes de 200, 300 y 400 µm.

Dado que las mediciones de un mismo participante pueden estar correlacionadas, no deben tratarse como observaciones independientes. Se utiliza un ANOVA de medidas repetidas para evaluar las diferencias del módulo de Young estimado entre longitudes, considerando al participante como unidad de repetición.

El análisis estadístico tiene un propósito exploratorio y está condicionado por el tamaño reducido de la muestra y la procedencia digitalizada de los datos.

In [9]:

# ============================================================
# 4.1. ESTRUCTURA DE LAS MEDICIONES REPETIDAS
# ============================================================

participant_summary = (
    df_marshall.groupby(["participant", "chip"])
    .agg(
        observations=("E_GPa", "size"),
        lengths=("L_um", "nunique"),
        mean_e_gpa=("E_GPa", "mean"),
        std_e_gpa=("E_GPa", "std"),
    )
    .reset_index()
)

print("Estructura experimental por participante:")
display(participant_summary.round(4))

print("\nParticipantes por chip:")
display(
    df_marshall.groupby("chip")["participant"]
    .nunique()
    .to_frame(name="Participantes")
)

print("\nLongitudes por participante:")
display(
    df_marshall.groupby("participant")["L_um"]
    .agg(lambda values: sorted(values.unique()))
    .to_frame(name="Longitudes (µm)")
)

# Validar el diseño de medidas repetidas.
expected_lengths = {200, 300, 400}

lengths_by_participant = df_marshall.groupby(
    "participant"
)["L_um"].agg(set)

assert len(lengths_by_participant) == 8
assert all(
    lengths == expected_lengths
    for lengths in lengths_by_participant
), "El diseño de medidas repetidas no está completo."

assert (
    df_marshall.groupby(["participant", "L_um"])
    .size()
    .eq(1)
    .all()
), "Se encontraron mediciones duplicadas por participante y longitud."

print("\nValidación del diseño experimental: correcta.")


Estructura experimental por participante:


,participant,chip,observations,lengths,mean_e_gpa,std_e_gpa
0,1,chip 1,3,3,64.6633,3.9559
1,2,chip 1,3,3,64.2600,3.4662
2,3,chip 1,3,3,64.3300,3.6306
3,4,chip 2,3,3,61.8733,3.2146
4,5,chip 2,3,3,61.5333,3.5945
5,6,chip 2,3,3,60.9200,3.8410
6,7,chip 3,3,3,61.8333,3.2949
7,8,chip 4,3,3,63.1500,4.7721



Participantes por chip:


,Participantes
chip,
chip 1,3
chip 2,3
chip 3,1
chip 4,1



Longitudes por participante:


,Longitudes (µm)
participant,
1,"[200, 300, 400]"
2,"[200, 300, 400]"
3,"[200, 300, 400]"
4,"[200, 300, 400]"
5,"[200, 300, 400]"
6,"[200, 300, 400]"
7,"[200, 300, 400]"
8,"[200, 300, 400]"



Validación del diseño experimental: correcta.


### 4.2. ANOVA de medidas repetidas

Se aplica un análisis de varianza (ANOVA) de medidas repetidas para evaluar si el módulo de Young estimado (`E_GPa`) presenta diferencias entre las tres longitudes del voladizo: 200, 300 y 400 µm.

**Hipótesis:**

- **H₀:** el promedio del módulo de Young estimado es igual para las tres longitudes.
- **H₁:** al menos una de las longitudes presenta un promedio diferente.

Se considera un nivel de significancia de α = 0.05.

El participante se utiliza como unidad de repetición para considerar la dependencia de las mediciones realizadas dentro de cada participante.

Posteriormente se examinarán los supuestos de normalidad y esfericidad, incluyendo la corrección de Greenhouse-Geisser cuando corresponda.

In [11]:

# ============================================================
# 4.2. ANOVA DE MEDIDAS REPETIDAS
# ============================================================

df_anova = df_marshall[
    ["participant", "L_um", "E_GPa"]
].copy()

# Tratar la longitud como factor categórico.
df_anova["L_um"] = df_anova["L_um"].astype(str)

anova_model = AnovaRM(
    data=df_anova,
    depvar="E_GPa",
    subject="participant",
    within=["L_um"],
)

anova_result = anova_model.fit()

print("Resultados del ANOVA de medidas repetidas:")
print(anova_result.summary())

print("\nEstadísticos principales:")
anova_table = anova_result.anova_table

f_statistic = anova_table.loc["L_um", "F Value"]
p_value = anova_table.loc["L_um", "Pr > F"]
df_num = anova_table.loc["L_um", "Num DF"]
df_den = anova_table.loc["L_um", "Den DF"]

print(f"F({df_num:.0f}, {df_den:.0f}) = {f_statistic:.4f}")
print(f"Valor p = {p_value:.8f}")
print(f"Significativo (α = 0.05): {p_value < 0.05}")


Resultados del ANOVA de medidas repetidas:
               Anova
     F Value  Num DF  Den DF Pr > F
-----------------------------------
L_um 294.2901 2.0000 14.0000 0.0000


Estadísticos principales:
F(2, 14) = 294.2901
Valor p = 0.00000000
Significativo (α = 0.05): True


### 4.3. Evaluación de normalidad mediante Shapiro-Wilk

Se aplica la prueba de Shapiro-Wilk para explorar la distribución del módulo de Young estimado en cada longitud del voladizo.

**Hipótesis:**

- **H₀:** los datos del grupo son compatibles con una distribución normal.
- **H₁:** los datos del grupo no siguen una distribución normal.

Se utiliza un nivel de significancia de α = 0.05.

Debido al tamaño reducido de cada grupo (n = 8), la prueba tiene una capacidad limitada para detectar desviaciones de normalidad. No rechazar la hipótesis nula no demuestra que los datos sean normales.

Además, como las mediciones corresponden a los mismos participantes, posteriormente se evaluará la normalidad de las diferencias entre longitudes, que resulta especialmente relevante para interpretar el diseño de medidas repetidas.

In [12]:

# ============================================================
# 4.3. NORMALIDAD POR LONGITUD
# ============================================================

shapiro_results = []

for length_um, group in df_marshall.groupby("L_um"):
    statistic, p_value_shapiro = shapiro(group["E_GPa"])

    shapiro_results.append(
        {
            "Longitud (µm)": length_um,
            "N": len(group),
            "Estadístico W": statistic,
            "Valor p": p_value_shapiro,
            "Decisión": (
                "No se rechaza H0"
                if p_value_shapiro >= 0.05
                else "Se rechaza H0"
            ),
        }
    )

df_shapiro = pd.DataFrame(shapiro_results)

print("Prueba de Shapiro-Wilk por longitud:")
display(
    df_shapiro.style.format(
        {
            "Estadístico W": "{:.4f}",
            "Valor p": "{:.6f}",
        }
    )
)


Prueba de Shapiro-Wilk por longitud:


,Longitud (µm),N,Estadístico W,Valor p,Decisión
0,200,8,0.8403,0.075821,No se rechaza H0
1,300,8,0.8885,0.226702,No se rechaza H0
2,400,8,0.8334,0.064403,No se rechaza H0


#### Interpretación de la normalidad por longitud

Los resultados de Shapiro-Wilk no proporcionan evidencia suficiente para rechazar la hipótesis de normalidad en ninguna de las tres longitudes evaluadas: 200 µm (W = 0.8403, p = 0.075821), 300 µm (W = 0.8885, p = 0.226702) y 400 µm (W = 0.8334, p = 0.064403).

Sin embargo, estos resultados deben interpretarse con cautela debido al tamaño muestral reducido (n = 8 por longitud). La ausencia de significancia estadística no garantiza que las distribuciones sean normales.

Por tanto, no se considera justificada una transformación automática de la variable objetivo mediante Box-Cox o Yeo-Johnson únicamente a partir de estas pruebas. La decisión deberá revisarse si se identifican problemas en los residuos de modelos posteriores.

Como siguiente comprobación, se evalúa la distribución de las diferencias entre las mediciones de un mismo participante, así como la esfericidad del diseño de medidas repetidas.

In [13]:

# ============================================================
# 4.4. NORMALIDAD DE DIFERENCIAS Y ESFERICIDAD
# ============================================================

df_wide = df_marshall.pivot(
    index="participant",
    columns="L_um",
    values="E_GPa",
)

length_pairs = [(200, 300), (200, 400), (300, 400)]
difference_results = []

for length_a, length_b in length_pairs:
    differences = df_wide[length_b] - df_wide[length_a]
    statistic, p_value_diff = shapiro(differences)

    difference_results.append(
        {
            "Comparación": f"{length_b} - {length_a} µm",
            "Diferencia media": differences.mean(),
            "Shapiro W": statistic,
            "Valor p": p_value_diff,
        }
    )

print("Normalidad de las diferencias entre longitudes:")
display(
    pd.DataFrame(difference_results).style.format(
        {
            "Diferencia media": "{:.4f}",
            "Shapiro W": "{:.4f}",
            "Valor p": "{:.6f}",
        }
    )
)

# Estimación del factor de corrección Greenhouse-Geisser.
covariance_matrix = np.cov(df_wide.to_numpy(), rowvar=False)
n_lengths = covariance_matrix.shape[0]

centering_matrix = (
    np.eye(n_lengths)
    - np.ones((n_lengths, n_lengths)) / n_lengths
)

centered_covariance = (
    centering_matrix
    @ covariance_matrix
    @ centering_matrix
)

epsilon_gg = np.trace(centered_covariance) ** 2 / (
    (n_lengths - 1)
    * np.trace(centered_covariance @ centered_covariance)
)

epsilon_gg = float(
    np.clip(epsilon_gg, 1 / (n_lengths - 1), 1.0)
)

corrected_df_num = df_num * epsilon_gg
corrected_df_den = df_den * epsilon_gg

corrected_p_value = f.sf(
    f_statistic,
    corrected_df_num,
    corrected_df_den,
)

print("\nCorrección de Greenhouse-Geisser:")
print(f"Epsilon: {epsilon_gg:.4f}")
print(
    "Grados de libertad corregidos: "
    f"({corrected_df_num:.4f}, {corrected_df_den:.4f})"
)
print(f"Valor p corregido: {corrected_p_value:.3e}")


Normalidad de las diferencias entre longitudes:


,Comparación,Diferencia media,Shapiro W,Valor p
0,300 - 200 µm,4.9675,0.9031,0.308170
1,400 - 200 µm,7.2475,0.8844,0.207560
2,400 - 300 µm,2.2800,0.9474,0.685165



Corrección de Greenhouse-Geisser:
Epsilon: 0.9100
Grados de libertad corregidos: (1.8200, 12.7402)
Valor p corregido: 3.105e-11


#### Interpretación de los supuestos y resultados del ANOVA

**Normalidad de las diferencias**

Las pruebas de Shapiro-Wilk aplicadas a las diferencias entre longitudes no permitieron rechazar la hipótesis de normalidad:

- 300 − 200 µm: W = 0.9031, p = 0.308170.
- 400 − 200 µm: W = 0.8844, p = 0.207560.
- 400 − 300 µm: W = 0.9474, p = 0.685165.

Estos resultados son compatibles con el supuesto de normalidad de las diferencias, aunque no lo demuestran debido al tamaño muestral reducido.

**Esfericidad y corrección de Greenhouse-Geisser**

Se obtuvo un factor epsilon de Greenhouse-Geisser de 0.9100, relativamente próximo a 1. Este resultado sugiere una desviación limitada respecto de la estructura de covarianza asociada con la esfericidad, pero no constituye por sí mismo una prueba formal de cumplimiento del supuesto.

Al aplicar la corrección, los grados de libertad se ajustaron a (1.8200, 12.7402) y se obtuvo un valor p de 3.105 × 10⁻¹¹.

**Resultado del ANOVA**

El ANOVA de medidas repetidas identificó diferencias estadísticamente significativas en el módulo de Young estimado entre las tres longitudes del voladizo: F(2, 14) = 294.2901, p < 0.001.

La conclusión se mantiene después de aplicar la corrección de Greenhouse-Geisser, por lo que el resultado es robusto frente a este ajuste de los grados de libertad.

Las diferencias medias observadas fueron:

- 300 − 200 µm: +4.9675 GPa.
- 400 − 200 µm: +7.2475 GPa.
- 400 − 300 µm: +2.2800 GPa.

Estas diferencias describen un aumento del módulo de Young estimado conforme aumenta la longitud en el conjunto analizado. No obstante, el ANOVA global no permite afirmar por sí solo que cada comparación individual sea estadísticamente significativa; para ello se requerirían contrastes pareados con corrección por comparaciones múltiples.

**Implicaciones para ingeniería de características**

Los resultados respaldan la relevancia de conservar la longitud del voladizo como característica explicativa. Sin embargo, las diferencias observadas pueden reflejar efectos asociados con el procedimiento experimental o la estimación del módulo de Young, y no necesariamente cambios en las propiedades intrínsecas del material.

Por tanto, la longitud se considera una característica relevante para etapas posteriores de modelado, manteniendo las precauciones correspondientes sobre dependencia entre mediciones y generalización.

### 4.5. Modelo mixto exploratorio

Se considera un modelo lineal mixto para explorar la relación del módulo de Young estimado con la longitud del voladizo y el chip utilizado.

La longitud y el chip se incluyen como efectos fijos, mientras que el participante se incorpora como intercepto aleatorio para representar la dependencia de sus mediciones repetidas.

Este análisis tiene carácter exploratorio debido al número reducido de participantes y al desequilibrio entre chips. En particular, los chips con un único participante no permiten distinguir adecuadamente los efectos asociados con el chip de los correspondientes al participante.

Se examinará la convergencia del ajuste y la varianza estimada del intercepto aleatorio. Si se presentan problemas de singularidad o estimaciones próximas a cero, los resultados no se utilizarán como evidencia confirmatoria de efectos independientes del chip.

In [16]:

# ============================================================
# 4.5. MODELO LINEAL MIXTO EXPLORATORIO
# ============================================================


df_mixed = df_marshall[
    ["participant", "chip", "L_um", "E_GPa"]
].copy()

df_mixed["length_factor"] = df_mixed["L_um"].astype(
    "category"
)

mixed_model = smf.mixedlm(
    "E_GPa ~ C(length_factor) + C(chip)",
    data=df_mixed,
    groups=df_mixed["participant"],
    eval_env=-1,
)

with warnings.catch_warnings(record=True) as model_warnings:
    warnings.simplefilter("always")

    try:
        mixed_result = mixed_model.fit(
            reml=True,
            method="lbfgs",
        )
    except (ValueError, np.linalg.LinAlgError) as error:
        mixed_result = None
        print(f"No fue posible ajustar el modelo: {error}")

if mixed_result is not None:
    print("Resumen del modelo mixto:")
    print(mixed_result.summary())

    print(f"\nConvergencia: {mixed_result.converged}")
    print(
        "Varianza del intercepto aleatorio: "
        f"{mixed_result.cov_re.iloc[0, 0]:.6f}"
    )

print("\nAdvertencias del ajuste:")
if model_warnings:
    for warning in model_warnings:
        print(f"- {warning.message}")
else:
    print("No se registraron advertencias.")


Resumen del modelo mixto:
               Mixed Linear Model Regression Results
Model:                 MixedLM     Dependent Variable:     E_GPa   
No. Observations:      24          Method:                 REML    
No. Groups:            8           Scale:                  0.3733  
Min. group size:       3           Log-Likelihood:         -21.7340
Max. group size:       3           Converged:              Yes     
Mean group size:       3.0                                         
-------------------------------------------------------------------
                        Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------------
Intercept               60.346    0.279 216.409 0.000 59.800 60.893
C(length_factor)[T.300]  4.967    0.306  16.260 0.000  4.369  5.566
C(length_factor)[T.400]  7.247    0.306  23.723 0.000  6.649  7.846
C(chip)[T.chip 2]       -2.976    0.305  -9.742 0.000 -3.574 -2.377
C(chip)[T.chip 3]       -2.584    0.4

/usr/local/lib/python3.13/dist-packages/statsmodels/formula/formulatools.py:70: FutureWarning: EvalEnvironment is deprecated and support be removed in a future version. You can
pass variables using a dict[str, value] where str is the variable name and the
value is the array of values.

  result = mgr.get_matrices(


#### Interpretación y limitaciones del modelo mixto

El modelo lineal mixto convergió utilizando el método REML, con 24 observaciones distribuidas entre ocho participantes y tres mediciones por participante.

Los coeficientes asociados con la longitud fueron positivos respecto de la categoría de referencia de 200 µm: +4.967 GPa para 300 µm y +7.247 GPa para 400 µm. Estos resultados son consistentes con las diferencias descriptivas identificadas previamente mediante el ANOVA de medidas repetidas.

No obstante, el ajuste generó una advertencia de singularidad en la matriz de covarianza de los efectos aleatorios. La varianza estimada del intercepto por participante fue de 0.015498, frente a una varianza residual de 0.3733. Esto indica que el componente aleatorio estimado es pequeño y que su identificación puede resultar inestable con los datos disponibles.

Aunque el modelo produjo coeficientes y valores p para los distintos chips, estos resultados no deben interpretarse como evidencia confirmatoria de efectos independientes del chip. La distribución de participantes entre chips es desigual, y los chips 3 y 4 cuentan con un solo participante cada uno. Además, la pertenencia al chip no varía dentro de cada participante, lo que limita la separación de ambos efectos.

**Decisión metodológica:** se conserva el modelo mixto únicamente como análisis exploratorio y de sensibilidad. El ANOVA de medidas repetidas permanece como análisis principal para evaluar las diferencias asociadas con la longitud. No se utilizarán los coeficientes del modelo mixto para establecer conclusiones causales sobre el efecto del chip.

Para una evaluación más sólida de las diferencias entre chips sería necesario contar con un mayor número de participantes independientes por chip y un diseño experimental que permita separar las fuentes de variabilidad.

### 4.6. Comparación exploratoria entre chips mediante la prueba de Welch

Se realiza una comparación exploratoria del módulo de Young estimado entre los chips 1 y 2, utilizando la prueba t de Welch para dos grupos independientes.

Para evitar la pseudorreplicación, primero se calcula el promedio de las tres mediciones de cada participante. De esta manera, la unidad de análisis es el participante y no cada registro individual.

Se comparan únicamente los chips 1 y 2, dado que ambos cuentan con tres participantes. Los chips 3 y 4 se excluyen de esta comparación porque solamente tienen un participante cada uno.

**Hipótesis:**

- **H₀:** el promedio del módulo de Young estimado es igual entre los participantes de los chips 1 y 2.
- **H₁:** los promedios son diferentes.

La prueba de Welch no requiere asumir igualdad de varianzas entre los grupos. Sin embargo, el reducido tamaño muestral (n = 3 por chip) limita la precisión de las estimaciones y la evaluación de sus supuestos.

Este análisis es descriptivo e inferencial exploratorio. No permite atribuir causalmente las diferencias observadas al chip utilizado.

In [17]:

# ============================================================
# 4.6. COMPARACIÓN EXPLORATORIA ENTRE CHIPS
# ============================================================

# Obtener una observación promedio por participante.
df_participants = (
    df_marshall.groupby(
        ["participant", "chip"],
        as_index=False,
    )
    .agg(mean_e_gpa=("E_GPa", "mean"))
)

print("Módulo de Young promedio por participante:")
display(df_participants.round(4))

print("\nResumen descriptivo por chip:")
chip_summary = (
    df_participants.groupby("chip")["mean_e_gpa"]
    .agg(["count", "mean", "std"])
)

display(chip_summary.round(4))

# Comparar chips con tres participantes independientes.
chip_1 = df_participants.loc[
    df_participants["chip"] == "chip 1",
    "mean_e_gpa",
]

chip_2 = df_participants.loc[
    df_participants["chip"] == "chip 2",
    "mean_e_gpa",
]

assert len(chip_1) == 3
assert len(chip_2) == 3

t_statistic, p_value_welch = ttest_ind(
    chip_1,
    chip_2,
    equal_var=False,
)

mean_difference = chip_1.mean() - chip_2.mean()

print("\nPrueba t de Welch: chip 1 frente a chip 2")
print(f"Participantes por grupo: {len(chip_1)} y {len(chip_2)}")
print(f"Estadístico t: {t_statistic:.4f}")
print(f"Valor p: {p_value_welch:.6f}")
print(f"Diferencia de medias: {mean_difference:.4f} GPa")


Módulo de Young promedio por participante:


,participant,chip,mean_e_gpa
0,1,chip 1,64.6633
1,2,chip 1,64.2600
2,3,chip 1,64.3300
3,4,chip 2,61.8733
4,5,chip 2,61.5333
5,6,chip 2,60.9200
6,7,chip 3,61.8333
7,8,chip 4,63.1500



Resumen descriptivo por chip:


,count,mean,std
chip,,,
chip 1,3,64.4178,0.2155
chip 2,3,61.4422,0.4832
chip 3,1,61.8333,NaN
chip 4,1,63.1500,NaN



Prueba t de Welch: chip 1 frente a chip 2
Participantes por grupo: 3 y 3
Estadístico t: 9.7418
Valor p: 0.003204
Diferencia de medias: 2.9756 GPa


#### Interpretación de la comparación entre chips

La prueba t de Welch identificó una diferencia estadísticamente significativa entre los promedios del módulo de Young estimado para los participantes asociados con los chips 1 y 2 (t = 9.7418, p = 0.003204).

El chip 1 presentó un promedio de 64.4178 GPa, mientras que el chip 2 presentó un promedio de 61.4422 GPa, con una diferencia de 2.9756 GPa a favor del primero.

Para evitar pseudorreplicación, el análisis utilizó una única observación promedio por participante, con tres participantes independientes por chip.

A pesar de la significancia estadística, los resultados deben interpretarse con cautela debido al tamaño muestral reducido y a la posible confusión entre los efectos del chip, el participante y las condiciones experimentales. La prueba de Welch no corrige estos posibles factores de confusión.

Los chips 3 y 4 no se incluyeron en la comparación inferencial porque cada uno cuenta con un solo participante, lo que impide estimar adecuadamente la variabilidad entre participantes dentro de esos grupos.

**Implicación para ingeniería de características:** la variable `chip` puede conservarse como información contextual del experimento. Sin embargo, su utilidad predictiva deberá evaluarse mediante una estrategia de validación que evite compartir participantes entre entrenamiento y evaluación.

En consecuencia, esta comparación se considera exploratoria y no constituye evidencia causal de que un chip produzca valores diferentes del módulo de Young.

### 5. Resumen de decisiones de ingeniería de características

Esta sección sintetiza las técnicas de ingeniería de características y análisis estadístico evaluadas sobre el conjunto de datos digitalizados de NIST Marshall, compuesto por 24 observaciones correspondientes a ocho participantes y tres longitudes de voladizo.

Se distinguen las técnicas efectivamente aplicadas, los análisis realizados con fines exploratorios y los procedimientos descartados o pendientes para etapas posteriores de modelado.

Las decisiones se fundamentan en la naturaleza de las variables, la estructura de medidas repetidas, el tamaño muestral y la necesidad de preservar la interpretabilidad física de los resultados.

La tabla siguiente proporciona una justificación metodológica de cada decisión y permite identificar las transformaciones que deberán evaluarse posteriormente dentro de un proceso de entrenamiento y validación sin fuga de información.

In [18]:

# ============================================================
# 5. RESUMEN DE DECISIONES DE INGENIERÍA DE CARACTERÍSTICAS
# ============================================================

decisiones_fe = pd.DataFrame(
    [
        {
            "Técnica": "One-hot encoding",
            "Decisión": "Aplicada",
            "Justificación": (
                "Se codificaron chip e instrumento como "
                "variables nominales sin orden natural."
            ),
        },
        {
            "Técnica": "Características físicas",
            "Decisión": "Aplicada de forma exploratoria",
            "Justificación": (
                "Se generaron L_mm, L_squared y L_inverse "
                "para explorar distintas representaciones "
                "de la longitud."
            ),
        },
        {
            "Técnica": "Estandarización",
            "Decisión": "Pendiente para modelado",
            "Justificación": (
                "No se aplicó escalamiento a la matriz final. "
                "Si el algoritmo lo requiere, se ajustará "
                "únicamente con los datos de entrenamiento "
                "para evitar fuga de información."
            ),
        },
        {
            "Técnica": "Selección de características",
            "Decisión": "Preselección exploratoria",
            "Justificación": (
                "Se excluyeron identificadores, coordenadas "
                "de digitalización, la variable objetivo y "
                "L_mm. Se conservaron temporalmente L_um, "
                "L_squared y L_inverse para analizar su "
                "redundancia antes del modelado."
            ),
        },
        {
            "Técnica": "Correlación de Pearson",
            "Decisión": "Aplicada",
            "Justificación": (
                "Se evaluó la asociación lineal con E_GPa "
                "y la redundancia entre características. "
                "Los resultados son exploratorios debido "
                "a las mediciones repetidas."
            ),
        },
        {
            "Técnica": "ANOVA de medidas repetidas",
            "Decisión": "Aplicada",
            "Justificación": (
                "Se evaluaron las diferencias de E_GPa "
                "entre longitudes respetando la dependencia "
                "de las mediciones por participante."
            ),
        },
        {
            "Técnica": "Shapiro-Wilk",
            "Decisión": "Aplicada",
            "Justificación": (
                "Se examinó la normalidad de E_GPa por "
                "longitud y de las diferencias pareadas."
            ),
        },
        {
            "Técnica": "Greenhouse-Geisser",
            "Decisión": "Aplicada",
            "Justificación": (
                "Se ajustaron los grados de libertad del "
                "ANOVA para evaluar la robustez del resultado "
                "frente a posibles desviaciones de esfericidad."
            ),
        },
        {
            "Técnica": "Modelo lineal mixto",
            "Decisión": "Exploratorio",
            "Justificación": (
                "Se analizaron longitud y chip considerando "
                "un intercepto aleatorio por participante. "
                "La advertencia de singularidad limita "
                "la interpretación de los coeficientes."
            ),
        },
        {
            "Técnica": "Prueba t de Welch",
            "Decisión": "Exploratoria",
            "Justificación": (
                "Se compararon chips 1 y 2 usando un "
                "promedio por participante para evitar "
                "pseudorreplicación. Los grupos contienen "
                "tres participantes cada uno."
            ),
        },
        {
            "Técnica": "Box-Cox / Yeo-Johnson",
            "Decisión": "No aplicada",
            "Justificación": (
                "No se identificó una justificación "
                "suficiente para transformar E_GPa "
                "en esta etapa exploratoria."
            ),
        },
        {
            "Técnica": "Chi-cuadrada",
            "Decisión": "No aplicada",
            "Justificación": (
                "La variable objetivo es continua y "
                "no se planteó una prueba de asociación "
                "entre variables categóricas."
            ),
        },
        {
            "Técnica": "Discretización (binning)",
            "Decisión": "No aplicada",
            "Justificación": (
                "Se conservaron los valores numéricos "
                "para evitar pérdida de resolución."
            ),
        },
        {
            "Técnica": "PCA",
            "Decisión": "No aplicada",
            "Justificación": (
                "Con 24 observaciones se priorizó "
                "la interpretabilidad de las "
                "características físicas."
            ),
        },
    ]
)

display(decisiones_fe)


,Técnica,Decisión,Justificación
0,One-hot encoding,Aplicada,Se codificaron chip e instrumento como variabl...
1,Características físicas,Aplicada de forma exploratoria,"Se generaron L_mm, L_squared y L_inverse para ..."
2,Estandarización,Pendiente para modelado,No se aplicó escalamiento a la matriz final. S...
3,Selección de características,Preselección exploratoria,"Se excluyeron identificadores, coordenadas de ..."
4,Correlación de Pearson,Aplicada,Se evaluó la asociación lineal con E_GPa y la ...
5,ANOVA de medidas repetidas,Aplicada,Se evaluaron las diferencias de E_GPa entre lo...
6,Shapiro-Wilk,Aplicada,Se examinó la normalidad de E_GPa por longitud...
7,Greenhouse-Geisser,Aplicada,Se ajustaron los grados de libertad del ANOVA ...
8,Modelo lineal mixto,Exploratorio,Se analizaron longitud y chip considerando un ...
9,Prueba t de Welch,Exploratoria,Se compararon chips 1 y 2 usando un promedio p...


### 6. Conclusiones generales del análisis NIST Marshall

El análisis de los 24 registros digitalizados del estudio NIST Marshall permitió explorar la relación entre la longitud de los voladizos, el módulo de Young estimado y las condiciones experimentales asociadas con los participantes, chips e instrumentos.

**Principales hallazgos**

1. **Relevancia de la longitud:** la longitud del voladizo presentó una asociación importante con el módulo de Young estimado. El ANOVA de medidas repetidas identificó diferencias estadísticamente significativas entre longitudes (F(2, 14) = 294.2901, p < 0.001), resultado que se mantuvo después de aplicar la corrección de Greenhouse-Geisser (ε = 0.9100, p < 0.001).

2. **Ingeniería de características:** se aplicó codificación one-hot a las variables categóricas y se generaron representaciones físicas de la longitud (`L_mm`, `L_squared` y `L_inverse`). El análisis de correlación mostró una alta redundancia entre las representaciones de longitud, por lo que será necesario seleccionar una representación adecuada durante el modelado.

3. **Supuestos estadísticos:** las pruebas de Shapiro-Wilk no identificaron evidencia suficiente para rechazar la normalidad en los grupos por longitud ni en las diferencias pareadas. Sin embargo, el tamaño reducido de la muestra limita la capacidad de estas pruebas para detectar desviaciones.

4. **Variabilidad entre chips:** la prueba de Welch identificó una diferencia estadísticamente significativa entre los promedios de los chips 1 y 2 (p = 0.003204). No obstante, el reducido número de participantes y la posible confusión con las condiciones experimentales impiden atribuir causalmente esta diferencia al chip.

5. **Limitaciones del modelo mixto:** aunque el modelo convergió, presentó una advertencia de singularidad en la covarianza de los efectos aleatorios. Por ello, sus coeficientes se conservaron únicamente como resultados exploratorios.

**Decisiones para etapas posteriores**

Se prioriza conservar la longitud como característica explicativa, evaluar la utilidad de las variables categóricas y evitar incluir simultáneamente representaciones altamente redundantes sin una justificación de modelado.

Cualquier estandarización o selección de características basada en los datos deberá realizarse dentro del procedimiento de entrenamiento y validación, evitando fugas de información. Asimismo, las particiones deberán respetar la agrupación por participante para impedir que mediciones relacionadas aparezcan simultáneamente en entrenamiento y evaluación.

**Conclusión final**

Los resultados respaldan la relevancia de la longitud del voladizo para explicar la variación observada en el módulo de Young estimado. Sin embargo, esta relación no debe interpretarse como evidencia de cambios intrínsecos en la rigidez del material, ya que puede estar influida por el procedimiento experimental y de estimación.

Este análisis establece una base estadística y metodológica para integrar las características seleccionadas en las siguientes etapas del proyecto, manteniendo la interpretabilidad física y las precauciones derivadas del tamaño y la estructura del conjunto de datos.